# Streaming

**Goal:** Stream responses and understand what UIs need from a streaming backend.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q anthropic

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the ANTHROPIC_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY'

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'  # good default: capable and cheap enough to iterate on


## Why streaming

Two latencies matter in an LLM product, and they're wildly different:

- **Time to first token (TTFT)** — how long until the user sees *anything*. Typically well under a couple of seconds.
- **Total latency** — how long until the response is complete. For a long answer, tens of seconds.

A non-streaming UI makes the user stare at a spinner for the total; a streaming UI feels responsive after the TTFT. Same model, same cost, entirely different product. Streaming is also the only practical way to run large `max_tokens` — the SDK refuses very large non-streaming requests because idle HTTP connections time out.

One 2026 note: `claude-sonnet-5` runs *adaptive thinking* by default, so the model may reason privately before the first visible token. Your measured TTFT includes that thinking time — worth knowing before you blame the network.


In [ ]:
import time

start = time.monotonic()
first_token_at = None

with client.messages.stream(
    model=MODEL,
    max_tokens=400,
    messages=[{'role': 'user', 'content': 'Explain, in ~150 words, why database indexes speed up reads but slow down writes.'}],
) as stream:
    for text in stream.text_stream:
        if first_token_at is None:
            first_token_at = time.monotonic() - start
        print(text, end='', flush=True)

total = time.monotonic() - start
print(f'\n\nTTFT: {first_token_at:.2f}s   total: {total:.2f}s')


Run it and note the gap between the two numbers — that gap is the user experience you buy with streaming. `text_stream` is the high-level convenience: it filters the event stream down to just text deltas. Production code usually needs the events underneath.

## The event stream anatomy

A streamed response is a typed sequence of server-sent events:

| Event | Fires | What's in it |
|---|---|---|
| `message_start` | once, first | message id, model, `usage.input_tokens` |
| `content_block_start` | per block | block index and type (`text`, `thinking`, `tool_use`) |
| `content_block_delta` | many | the increment: `text_delta`, `thinking_delta`, or `input_json_delta` |
| `content_block_stop` | per block | block finished |
| `message_delta` | near end | `stop_reason` and **final `usage.output_tokens`** |
| `message_stop` | once, last | stream is done |

Input tokens arrive at the *start*, output tokens at the *end* — remember that split when we get to billing.


In [ ]:
# Same request, raw events this time. We count deltas instead of printing every one.
from collections import Counter

event_counts = Counter()

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    messages=[{'role': 'user', 'content': 'Two sentences: what is connection pooling?'}],
) as stream:
    for event in stream:
        event_counts[event.type] += 1
        if event.type == 'message_start':
            print(f'message_start: model={event.message.model}, '
                  f'input_tokens={event.message.usage.input_tokens}')
        elif event.type == 'content_block_start':
            print(f'content_block_start: index={event.index}, type={event.content_block.type}')
        elif event.type == 'message_delta':
            print(f'message_delta: stop_reason={event.delta.stop_reason}, '
                  f'output_tokens={event.usage.output_tokens}')

    final = stream.get_final_message()

print()
print(dict(event_counts))
print('final usage:', final.usage)


Run it and note two things. First, you may see a `thinking` block start before the `text` block — that's adaptive thinking; its deltas carry empty text by default, so a UI should render it as a "thinking..." state, not silence. Second, `stream.get_final_message()` hands you the fully assembled `Message` at the end — you get incremental display *and* the complete object without gluing deltas together yourself.

## Streaming + tool use

Tool calls stream too. The tool's arguments arrive as `input_json_delta` events — fragments of a JSON string you accumulate per block index and parse when the block stops. This is how UIs show "Searching for: berlin weath..." while the model is still writing the call.


In [ ]:
import json

WEATHER_TOOL = {
    'name': 'get_weather',
    'description': 'Get current weather for a city. Call for any weather question.',
    'input_schema': {
        'type': 'object',
        'properties': {'city': {'type': 'string'}},
        'required': ['city'],
    },
}

partial_json = {}   # block index -> accumulated JSON string
tool_blocks = {}    # block index -> tool name

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    tools=[WEATHER_TOOL],
    messages=[{'role': 'user', 'content': 'What is the weather in Berlin?'}],
) as stream:
    for event in stream:
        if event.type == 'content_block_start' and event.content_block.type == 'tool_use':
            tool_blocks[event.index] = event.content_block.name
            partial_json[event.index] = ''
            print(f'tool call started: {event.content_block.name}')
        elif event.type == 'content_block_delta' and event.delta.type == 'input_json_delta':
            partial_json[event.index] += event.delta.partial_json
            print(f'  partial input so far: {partial_json[event.index]!r}')
        elif event.type == 'content_block_stop' and event.index in partial_json:
            args = json.loads(partial_json[event.index] or '{}')
            print(f'tool call complete: {tool_blocks[event.index]}({args})')

    final = stream.get_final_message()

# Sanity check: the SDK-assembled message has the same parsed input.
for block in final.content:
    if block.type == 'tool_use':
        print('from final message:', block.name, block.input)


Run it and note the `partial_json` fragments — they are *not* valid JSON until the block stops, so never `json.loads` mid-stream. In practice you accumulate manually only when the UI needs live argument display; otherwise `get_final_message()` gives you the parsed `block.input` and you continue the tool loop exactly as in the previous notebook (execute, append `tool_result`, open a new stream).

## What a real backend needs

In production you're rarely printing to a terminal — you're sitting between the model and a browser. Three concerns the notebook can't show but you should design for:

**1. Forwarding as SSE.** Don't buffer the whole response server-side; re-emit deltas as your own server-sent events. Sketch (FastAPI-flavored pseudocode — don't run this cell shape in Colab):

```python
@app.post('/chat')
async def chat(req: ChatRequest):
    async def gen():
        async with client.messages.stream(model=MODEL, max_tokens=1000,
                                          messages=req.messages) as stream:
            try:
                async for text in stream.text_stream:
                    yield f'data: {json.dumps({"delta": text})}\n\n'
            finally:
                # Runs even if the client disconnected mid-stream:
                final = await stream.get_final_message()
                record_usage(req.user_id, final.usage)   # billing truth
        yield 'data: [DONE]\n\n'
    return StreamingResponse(gen(), media_type='text/event-stream')
```

**2. Client disconnects.** Users close tabs mid-answer constantly. Your generator gets cancelled — but the model kept generating and you keep paying. Decide deliberately: abort the upstream request on disconnect (saves tokens, loses the answer) or let it finish and persist the result (costs tokens, enables "resume"). Either way, the `finally` block must still capture usage.

**3. Usage for billing.** Output-token counts arrive in the final `message_delta` — if you only forward text deltas and drop the tail events, you've thrown away the bill. Always read the final message (or the `message_delta` event) server-side and record it before the request handler exits.


## Exercises

1. Extend the TTFT cell to also record *inter-token gaps* (time between consecutive deltas) and print p50/p95. This is the number that makes streamed output feel smooth or janky.
2. Build `stream_to_list(prompt)` that returns `(chunks, final_usage)` — every text delta in order plus the final usage object — using only raw events (no `text_stream`, no `get_final_message`). Verify the joined chunks equal the final message text.
3. Combine this notebook with the previous one: a `run_agent_streaming` loop that streams each turn, prints text deltas live, then executes tool calls from `get_final_message()` and continues until `stop_reason == 'end_turn'`.
4. Simulate a client disconnect: break out of the event loop after the 10th text delta, and confirm you can still recover usage via `get_final_message()` inside the context manager. Then check what happens if you exit the `with` block instead.
